# Ejercicio 7: indicadores

Ejecuta las consultas de `sql/ejercicio7/` sobre `data/processed/tablero.duckdb`, las mismas que usa el tablero de Metabase. Requiere haber ejecutado `scripts/construir_tablero.py`. La interpretación está en `docs/ejercicio7.md`.

In [1]:
import os
from pathlib import Path

import duckdb
import pandas as pd

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
DIR_SQL = Path("sql/ejercicio7")

# Solo lectura, igual que Metabase
con = duckdb.connect("data/processed/tablero.duckdb", read_only=True)


def ejecutar(nombre: str) -> pd.DataFrame:
    consulta = (DIR_SQL / f"{nombre}.sql").read_text()
    print(consulta)
    return con.sql(consulta).df()

## Base del tablero

In [2]:
con.sql("""
    SELECT table_name AS objeto, 'tabla' AS clase, estimated_size AS filas, column_count AS columnas
    FROM duckdb_tables()
    UNION ALL
    SELECT view_name, 'vista', NULL, column_count FROM duckdb_views() WHERE NOT internal
    ORDER BY clase, objeto
""").df()

,objeto,clase,filas,columnas
0,calidad_mensual,tabla,40,10
1,viajes,tabla,69256941,16
2,zonas,tabla,265,4
3,viajes_comparables,vista,<NA>,16


Cuadre: registros de los Parquet menos excluidos debe ser igual a las filas de `viajes`.

In [3]:
con.sql("""
    SELECT c.tipo, c.anio, c.registros, c.excluidos, v.viajes,
           c.registros - c.excluidos = v.viajes AS cuadra,
           round(100 * c.excluidos / c.registros, 2) AS pct_excluidos
    FROM (SELECT tipo, anio, sum(registros)::BIGINT AS registros, sum(excluidos)::BIGINT AS excluidos
          FROM calidad_mensual GROUP BY ALL) c
    JOIN (SELECT tipo, anio, count(*) AS viajes FROM viajes GROUP BY ALL) v USING (tipo, anio)
    ORDER BY c.anio, c.tipo DESC
""").df()

,tipo,anio,registros,excluidos,viajes,cuadra,pct_excluidos
0,yellow,2024,41169720,1459760,39709960,True,3.55
1,green,2024,660218,36535,623683,True,5.53
2,yellow,2026,29703355,1104123,28599232,True,3.72
3,green,2026,337114,13048,324066,True,3.87


Meses de `viajes_comparables`:

In [4]:
con.sql("SELECT anio, min(mes) AS primer_mes, max(mes) AS ultimo_mes, count(DISTINCT mes) AS meses FROM viajes_comparables GROUP BY anio ORDER BY anio").df()

,anio,primer_mes,ultimo_mes,meses
0,2024,1,8,8
1,2026,1,8,8


## I1. Resumen por tipo y año

In [5]:
ejecutar("i01_resumen_anual")

-- I1. Resumen por tipo y año
-- Pregunta: ¿qué tamaño tiene cada servicio y cómo cambia entre años?
SELECT tipo,
       anio::VARCHAR AS anio,
       min(mes) || ' a ' || max(mes) AS meses,
       round(count(*) / count(DISTINCT pickup::DATE), 0) AS viajes_por_dia,
       round(sum(total) / count(DISTINCT pickup::DATE) / 1e3, 1) AS ingresos_por_dia_miles_usd,
       round(median(total), 2) AS total_mediano_usd,
       round(median(distancia_mi), 2) AS distancia_mediana_mi,
       round(median(duracion_min), 1) AS duracion_mediana_min
FROM viajes_comparables
GROUP BY tipo, anio
ORDER BY tipo DESC, anio;



,tipo,anio,meses,viajes_por_dia,ingresos_por_dia_miles_usd,total_mediano_usd,distancia_mediana_mi,duracion_mediana_min
0,yellow,2024,1 a 8,104567.0,2962.4,21.00,1.80,12.7
1,yellow,2026,1 a 8,117692.0,3554.6,23.58,1.92,14.1
2,green,2024,1 a 8,1716.0,40.9,19.15,1.96,11.9
3,green,2026,1 a 8,1334.0,34.0,20.52,2.15,13.4


## I2. Viajes por día en cada mes

In [6]:
ejecutar("i02a_demanda_yellow").pivot(index="mes", columns="anio", values="viajes_por_dia")

-- I2a. Viajes por día en cada mes, taxis amarillos
-- Pregunta: ¿cómo evoluciona la demanda durante el año y entre años?
SELECT mes,
       anio::VARCHAR AS anio,       -- texto, para que Metabase lo use como serie
       round(count(*) / day(last_day(make_date(anio, mes, 1))), 0) AS viajes_por_dia
FROM viajes
WHERE tipo = 'yellow'
GROUP BY anio, mes
ORDER BY anio, mes;



anio,2024,2026
mes,,
1,92575.0,114888.0
2,100057.0,116106.0
3,110979.0,122942.0
4,113819.0,124124.0
5,116709.0,127869.0
6,114282.0,123198.0
7,95935.0,109286.0
8,92497.0,103358.0
9,116134.0,NaN


In [7]:
ejecutar("i02b_demanda_green").pivot(index="mes", columns="anio", values="viajes_por_dia")

-- I2b. Viajes por día en cada mes, taxis verdes
-- Pregunta: ¿cómo evoluciona la demanda durante el año y entre años?
SELECT mes,
       anio::VARCHAR AS anio,       -- texto, para que Metabase lo use como serie
       round(count(*) / day(last_day(make_date(anio, mes, 1))), 0) AS viajes_por_dia
FROM viajes
WHERE tipo = 'green'
GROUP BY anio, mes
ORDER BY anio, mes;



anio,2024,2026
mes,,
1,1726.0,1254.0
2,1744.0,1282.0
3,1751.0,1376.0
4,1770.0,1418.0
5,1862.0,1396.0
6,1729.0,1419.0
7,1570.0,1274.0
8,1577.0,1251.0
9,1717.0,NaN


## I3. Perfil horario

In [8]:
ejecutar("i03_perfil_horario").pivot(index="hora", columns="serie", values="pct_viajes")

-- I3. Perfil horario de la demanda
-- Pregunta: ¿a qué horas se usa cada servicio?
-- Porcentaje dentro de cada tipo y franja: compara la forma del perfil, no el volumen.
SELECT hora,
       tipo || CASE WHEN dia_semana <= 5 THEN ' laboral' ELSE ' fin de semana' END AS serie,
       round(100 * count(*) / sum(count(*)) OVER (PARTITION BY tipo, dia_semana <= 5), 2) AS pct_viajes
FROM viajes
GROUP BY hora, tipo, dia_semana <= 5
ORDER BY serie, hora;



serie,green fin de semana,green laboral,yellow fin de semana,yellow laboral
hora,,,,
0,3.07,1.25,5.61,1.98
1,2.41,0.72,4.45,0.99
2,2.05,0.41,3.17,0.55
3,1.74,0.26,2.17,0.37
4,1.30,0.30,1.34,0.42
5,0.75,0.57,0.66,0.79
6,0.90,1.94,0.93,1.79
7,1.65,4.51,1.30,3.51
8,2.34,5.70,1.97,4.71


## I4. Velocidad mediana por hora en el centro de Manhattan

In [9]:
ejecutar("i04_velocidad_hora").pivot(index="hora", columns="anio", values="velocidad_mediana_mph")

-- I4. Velocidad mediana por hora en el centro de Manhattan (yellow, días laborales)
-- Pregunta: ¿cómo cambia la congestión durante el día y entre años?
-- Origen y destino en la Yellow Zone, para que un cambio de zonas no se confunda con tráfico.
SELECT v.hora,
       v.anio::VARCHAR AS anio,
       round(median(v.velocidad_mph), 2) AS velocidad_mediana_mph
FROM viajes_comparables v
JOIN zonas zo ON v.origen_id = zo.zona_id
JOIN zonas zd ON v.destino_id = zd.zona_id
WHERE v.tipo = 'yellow'
  AND v.dia_semana <= 5
  AND v.velocidad_mph IS NOT NULL
  AND zo.service_zone = 'Yellow Zone'
  AND zd.service_zone = 'Yellow Zone'
GROUP BY v.hora, v.anio
ORDER BY anio, v.hora;



anio,2024,2026
hora,,
0,11.80,11.29
1,12.46,12.08
2,12.73,12.55
3,13.09,12.93
4,13.70,13.53
5,13.48,13.35
6,12.59,12.14
7,10.32,9.96
8,8.39,8.19


## I5. Total mediano según el canal

In [10]:
ejecutar("i05_costo_viaje").pivot(index="grupo", columns="canal", values="total_mediano_usd")

-- I5. Total mediano por viaje según el canal
-- Pregunta: ¿cuánto cuesta un viaje y cuánto más cuesta pedirlo por aplicación?
-- Se usa el total porque los componentes de los viajes por aplicación están incompletos (Ej. 4, P13).
SELECT tipo || ' ' || anio AS grupo,
       CASE WHEN forma_pago IN ('Flex Fare', 'Sin dato') THEN 'Aplicación' ELSE 'Taxímetro' END AS canal,
       round(median(total), 2) AS total_mediano_usd
FROM viajes_comparables
WHERE forma_pago IN ('Tarjeta', 'Efectivo', 'Flex Fare', 'Sin dato')
GROUP BY tipo, anio, canal
ORDER BY tipo DESC, anio, canal DESC;



canal,Aplicación,Taxímetro
grupo,,
green 2024,25.18,18.90
green 2026,26.47,19.90
yellow 2024,22.25,20.93
yellow 2026,28.99,21.90


## I6. Formas de pago

In [11]:
ejecutar("i06_formas_pago").pivot(index="grupo", columns="forma", values="pct_viajes")

-- I6. Formas de pago
-- Pregunta: ¿cómo pagan los pasajeros y cómo cambia con el tiempo?
-- 'Aplicación' junta Flex Fare (yellow) y los viajes sin forma de pago de green (Ej. 4, P11).
SELECT tipo || ' ' || anio AS grupo,
       CASE WHEN forma_pago IN ('Flex Fare', 'Sin dato') THEN 'Aplicación'
            WHEN forma_pago IN ('Tarjeta', 'Efectivo') THEN forma_pago
            ELSE 'Otros' END AS forma,
       round(100 * count(*) / sum(count(*)) OVER (PARTITION BY tipo, anio), 2) AS pct_viajes
FROM viajes_comparables
GROUP BY tipo, anio, forma
ORDER BY tipo DESC, anio,
         list_position(['Tarjeta', 'Efectivo', 'Aplicación', 'Otros'], forma);



forma,Aplicación,Efectivo,Otros,Tarjeta
grupo,,,,
green 2024,4.01,27.78,0.45,67.75
green 2026,14.71,19.41,0.34,65.54
yellow 2024,8.99,13.79,1.30,75.92
yellow 2026,24.62,9.10,0.63,65.65


## I7. Propinas

In [12]:
ejecutar("i07_propinas").pivot(index="grupo", columns="tramo", values="pct_viajes")

-- I7. Propina sobre la tarifa, pagos con tarjeta
-- Pregunta: ¿cuánta propina se deja y cómo cambia?
-- Solo tarjeta: las propinas en efectivo no se registran (Ej. 4, P9).
WITH t AS (
    SELECT tipo, anio, 100 * propina / tarifa AS pct
    FROM viajes_comparables
    WHERE payment_type = 1 AND tarifa > 0
)
SELECT tipo || ' ' || anio AS grupo,
       CASE WHEN pct = 0 THEN '1. sin propina'
            WHEN pct < 20 THEN '2. menos de 20%'
            WHEN pct < 25 THEN '3. 20% a 25%'
            WHEN pct < 30 THEN '4. 25% a 30%'
            ELSE '5. 30% o más' END AS tramo,
       round(100 * count(*) / sum(count(*)) OVER (PARTITION BY tipo, anio), 2) AS pct_viajes
FROM t
GROUP BY tipo, anio, tramo
ORDER BY tipo DESC, anio, tramo;



tramo,1. sin propina,2. menos de 20%,3. 20% a 25%,4. 25% a 30%,5. 30% o más
grupo,,,,,
green 2024,8.81,20.45,30.40,22.85,17.49
green 2026,8.66,19.06,30.93,23.68,17.66
yellow 2024,5.56,19.16,19.34,27.69,28.25
yellow 2026,8.79,17.42,15.88,25.73,32.18


## I8. Viajes de aeropuerto

In [13]:
ejecutar("i08_aeropuertos")

-- I8. Peso de los viajes de aeropuerto
-- Pregunta: ¿cuánto aportan los aeropuertos en viajes y en ingresos?
-- Aeropuerto: origen o destino en JFK (132), LaGuardia (138) o Newark (1).
SELECT tipo || ' ' || anio AS grupo,
       round(100 * avg((origen_id IN (1, 132, 138) OR destino_id IN (1, 132, 138))::INT), 2) AS pct_viajes,
       round(100 * sum(total) FILTER (WHERE origen_id IN (1, 132, 138) OR destino_id IN (1, 132, 138))
             / sum(total), 2) AS pct_ingresos
FROM viajes_comparables
GROUP BY tipo, anio
ORDER BY tipo DESC, anio;



,grupo,pct_viajes,pct_ingresos
0,yellow 2024,10.16,28.04
1,yellow 2026,8.15,21.05
2,green 2024,4.15,8.09
3,green 2026,3.34,6.35


## I9. Zona de origen

In [14]:
ejecutar("i09_zona_origen").pivot(index="grupo", columns="origen", values="pct_viajes")

-- I9. Zona de origen
-- Pregunta: ¿dónde opera cada servicio y se mantiene su separación?
SELECT v.tipo || ' ' || v.anio AS grupo,
       CASE WHEN z.service_zone = 'Yellow Zone' THEN 'Manhattan centro (Yellow Zone)'
            WHEN z.service_zone = 'Boro Zone' AND z.borough = 'Manhattan' THEN 'Alto Manhattan'
            WHEN z.service_zone = 'Boro Zone' AND z.borough IN ('Brooklyn', 'Queens') THEN z.borough
            -- aeropuertos (ver I8), Bronx, Staten Island y zonas desconocidas
            ELSE 'Resto' END AS origen,
       round(100 * count(*) / sum(count(*)) OVER (PARTITION BY v.tipo, v.anio), 2) AS pct_viajes
FROM viajes_comparables v
JOIN zonas z ON v.origen_id = z.zona_id
GROUP BY v.tipo, v.anio, origen
ORDER BY v.tipo DESC, v.anio,
         list_position(['Manhattan centro (Yellow Zone)', 'Alto Manhattan', 'Brooklyn', 'Queens', 'Resto'], origen);



origen,Alto Manhattan,Brooklyn,Manhattan centro (Yellow Zone),Queens,Resto
grupo,,,,,
green 2024,53.28,13.40,7.16,24.87,1.29
green 2026,52.51,15.81,6.95,22.01,2.72
yellow 2024,2.06,1.25,86.85,1.25,8.58
yellow 2026,3.49,3.56,83.23,2.30,7.42


## I10. Calidad mensual

In [15]:
ejecutar("i10_calidad_mensual").pivot(index="mes", columns="tipo", values="pct_excluidos")

-- I10. Registros excluidos por las reglas de calidad
-- Pregunta: ¿qué tan confiables son los datos de cada mes y tipo?
SELECT make_date(anio, mes, 1) AS mes,
       tipo,
       round(100 * sum(excluidos) / sum(registros), 2) AS pct_excluidos
FROM calidad_mensual
GROUP BY ALL
ORDER BY tipo DESC, mes;



tipo,green,yellow
mes,,
2024-01-01,5.39,3.20
2024-02-01,5.59,3.52
2024-03-01,5.55,3.97
2024-04-01,5.95,2.84
2024-05-01,5.40,2.84
2024-06-01,5.27,3.13
2024-07-01,6.10,3.35
2024-08-01,5.55,3.75
2024-09-01,5.41,4.10
